# E6 - Inventario dataset axial Al-Kafri/Sudirman

Este notebook realiza un inventario controlado del dataset complementario Al-Kafri/Sudirman para evaluar su utilidad como fuente axial nativa de RM lumbar.

Alcance:

- No entrena modelos.
- No crea baseline.
- No modifica SPIDER.
- No sube datos al repositorio.
- No borra ZIPs originales.
- Solo descomprime dentro de `/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/extracted/`.
- Genera CSVs, figuras, JSON y conclusion tecnica para decidir el preprocesamiento axial del notebook 13.

In [ ]:
# Dependencias para Google Colab.
try:
    import pydicom  # noqa: F401
except Exception:
    %pip install -q pydicom

In [ ]:
import json
import os
import shutil
import time
import zipfile
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pydicom
from PIL import Image
from scipy.io import whosmat
from tqdm.auto import tqdm

pd.set_option("display.max_columns", 160)

## Montaje de Drive y rutas

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as exc:
    print("No se monto Google Drive automaticamente. Si estas en Colab, montalo manualmente.")
    print("Detalle:", repr(exc))

In [ ]:
ALKAFRI_ROOT = Path("/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI")
RAW_ZIPS = ALKAFRI_ROOT / "raw_zips"
EXTRACTED_ROOT = ALKAFRI_ROOT / "extracted"
INVENTORY_ROOT = ALKAFRI_ROOT / "inventory"
PROCESSED_ROOT = ALKAFRI_ROOT / "processed"
RESULTS_ROOT = Path("/content/drive/MyDrive/PFI_MVP/results/E6_alkafri_inventory")
FIGURES_ROOT = Path("/content/drive/MyDrive/PFI_MVP/figures")
DOCS_ROOT = Path("/content/drive/MyDrive/PFI_MVP/docs")

for path in [RAW_ZIPS, EXTRACTED_ROOT, INVENTORY_ROOT, PROCESSED_ROOT, RESULTS_ROOT, FIGURES_ROOT, DOCS_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

if not ALKAFRI_ROOT.exists():
    raise FileNotFoundError(f"No existe ALKAFRI_ROOT: {ALKAFRI_ROOT}")

FORCE_EXTRACT = False

EXPECTED_ZIPS = {
    "k57fr854j2-2.zip": "main_dataset",
    "Label Image Ground Truth Data for Lumbar Spine MRI Dataset.zip": "ground_truth",
    "Source_Code.zip": "source_code",
    "Radiologists Notes for Lumbar Spine MRI Dataset.zip": "radiologist_notes",
}

print("ALKAFRI_ROOT:", ALKAFRI_ROOT)
print("RAW_ZIPS:", RAW_ZIPS)
print("EXTRACTED_ROOT:", EXTRACTED_ROOT)
print("FORCE_EXTRACT:", FORCE_EXTRACT)

## Verificacion de ZIPs

In [ ]:
def human_size(num_bytes):
    value = float(num_bytes)
    for unit in ["B", "KB", "MB", "GB", "TB"]:
        if value < 1024 or unit == "TB":
            return f"{value:.2f} {unit}"
        value /= 1024


zip_paths = sorted(RAW_ZIPS.glob("*.zip"))
zip_rows = []
for path in zip_paths:
    zip_rows.append({
        "zip_name": path.name,
        "zip_path": str(path),
        "expected": path.name in EXPECTED_ZIPS,
        "target_folder": EXPECTED_ZIPS.get(path.name, path.stem),
        "size_bytes": int(path.stat().st_size),
        "size_human": human_size(path.stat().st_size),
    })

zip_files_df = pd.DataFrame(zip_rows)
zip_files_csv_path = RESULTS_ROOT / "E6_alkafri_zip_files.csv"
zip_files_df.to_csv(zip_files_csv_path, index=False)

missing_expected = sorted(set(EXPECTED_ZIPS) - {path.name for path in zip_paths})
print("ZIPs detectados:", len(zip_paths))
print("ZIPs esperados faltantes:", missing_expected)
print("CSV ZIPs:", zip_files_csv_path)
display(zip_files_df)

if missing_expected:
    raise FileNotFoundError(f"Faltan ZIPs esperados: {missing_expected}")

## Inventario interno de ZIPs sin descomprimir

In [ ]:
KEYWORDS = [
    "axial", "sagittal", "sag", "mask", "label", "ground", "gt", "segmentation",
    "l3", "l4", "l5", "s1",
]
TRACKED_EXTENSIONS = [".ima", ".dcm", ".png", ".jpg", ".jpeg", ".bmp", ".mat", ".csv", ".txt", ".m"]


def extension_group(name):
    ext = Path(name).suffix.lower()
    return ext if ext in TRACKED_EXTENSIONS else "other"


def keyword_flags(text):
    text_lower = str(text).lower()
    return {f"has_{kw}": bool(kw in text_lower) for kw in KEYWORDS}


zip_inventory_rows = []
for zip_path in tqdm(zip_paths, desc="Inventariando ZIPs"):
    with zipfile.ZipFile(zip_path, "r") as zf:
        infos = [info for info in zf.infolist() if not info.is_dir()]
        names = [info.filename for info in infos]
        ext_counts = Counter(extension_group(name) for name in names)
        all_text = " ".join(names).lower()
        examples = names[:20]

        row = {
            "zip_name": zip_path.name,
            "zip_path": str(zip_path),
            "target_folder": EXPECTED_ZIPS.get(zip_path.name, zip_path.stem),
            "total_files": int(len(infos)),
            "example_internal_paths": json.dumps(examples, ensure_ascii=False),
        }
        for ext in TRACKED_EXTENSIONS + ["other"]:
            row[f"count_{ext.replace('.', '')}"] = int(ext_counts.get(ext, 0))
        row.update(keyword_flags(all_text))
        zip_inventory_rows.append(row)

zip_inventory_df = pd.DataFrame(zip_inventory_rows)
zip_inventory_csv_path = RESULTS_ROOT / "E6_alkafri_zip_inventory.csv"
zip_inventory_df.to_csv(zip_inventory_csv_path, index=False)

print("Inventario ZIP:", zip_inventory_csv_path)
display(zip_inventory_df)

## Descompresion controlada

In [ ]:
def safe_extract_zip(zip_path, destination):
    destination = Path(destination).resolve()
    with zipfile.ZipFile(zip_path, "r") as zf:
        for member in zf.infolist():
            target_path = (destination / member.filename).resolve()
            if not str(target_path).startswith(str(destination)):
                raise RuntimeError(f"Ruta insegura dentro del ZIP: {member.filename}")
        zf.extractall(destination)


extraction_rows = []
for zip_path in tqdm(zip_paths, desc="Extrayendo ZIPs"):
    target_folder = EXPECTED_ZIPS.get(zip_path.name, zip_path.stem)
    destination = EXTRACTED_ROOT / target_folder
    destination.mkdir(parents=True, exist_ok=True)
    existing_files = [p for p in destination.rglob("*") if p.is_file()]
    skipped = bool(existing_files and not FORCE_EXTRACT)
    start = time.time()

    if skipped:
        extracted_count = len(existing_files)
        elapsed_seconds = 0.0
    else:
        if FORCE_EXTRACT and destination.exists():
            shutil.rmtree(destination)
            destination.mkdir(parents=True, exist_ok=True)
        safe_extract_zip(zip_path, destination)
        extracted_count = sum(1 for p in destination.rglob("*") if p.is_file())
        elapsed_seconds = time.time() - start

    extraction_rows.append({
        "zip_name": zip_path.name,
        "target_folder": target_folder,
        "destination": str(destination),
        "skipped_existing": skipped,
        "force_extract": FORCE_EXTRACT,
        "extracted_file_count": int(extracted_count),
        "elapsed_seconds": float(elapsed_seconds),
    })

extraction_report_df = pd.DataFrame(extraction_rows)
extraction_report_csv_path = RESULTS_ROOT / "E6_alkafri_extraction_report.csv"
extraction_report_df.to_csv(extraction_report_csv_path, index=False)

print("Reporte extraccion:", extraction_report_csv_path)
display(extraction_report_df)

## Inventario posterior a la extraccion

In [ ]:
def probable_source_from_path(path):
    parts = {part.lower() for part in Path(path).parts}
    for source in ["main_dataset", "ground_truth", "source_code", "radiologist_notes"]:
        if source in parts:
            return source
    return "unknown"


def probable_type_from_path(path):
    path = Path(path)
    ext = path.suffix.lower()
    text = str(path).lower()
    if ext == ".ima":
        return "image_dicom_ima"
    if ext == ".dcm":
        return "image_dicom_dcm"
    if ext in {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"} and any(k in text for k in ["mask", "label", "ground", "gt", "seg"]):
        return "mask_or_label_image"
    if ext in {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}:
        return "image_file"
    if ext == ".mat":
        return "matlab_file"
    if ext in {".txt", ".csv", ".xlsx", ".xls", ".doc", ".docx"}:
        return "text_metadata"
    return "unknown"


file_rows = []
for path in tqdm([p for p in EXTRACTED_ROOT.rglob("*") if p.is_file()], desc="Inventario extraido"):
    rel = path.relative_to(EXTRACTED_ROOT)
    row = {
        "file_path": str(path),
        "relative_path": str(rel),
        "file_name": path.name,
        "extension": path.suffix.lower(),
        "parent_folder": path.parent.name,
        "size_bytes": int(path.stat().st_size),
        "probable_source": probable_source_from_path(path),
        "probable_type": probable_type_from_path(path),
    }
    row.update(keyword_flags(str(rel)))
    file_rows.append(row)

extracted_inventory_df = pd.DataFrame(file_rows)
extracted_inventory_csv_path = RESULTS_ROOT / "E6_alkafri_extracted_file_inventory.csv"
extracted_inventory_df.to_csv(extracted_inventory_csv_path, index=False)

print("Inventario extraido:", extracted_inventory_csv_path)
print("Archivos inventariados:", len(extracted_inventory_df))
display(extracted_inventory_df.head())

## Estructura de carpetas y extensiones

In [ ]:
def summarize_tree(root, max_depth=3, max_entries=250):
    lines = []
    root = Path(root)
    for path in sorted(root.rglob("*")):
        rel = path.relative_to(root)
        depth = len(rel.parts)
        if depth > max_depth:
            continue
        indent = "  " * (depth - 1)
        suffix = "/" if path.is_dir() else ""
        lines.append(f"{indent}{rel.name}{suffix}")
        if len(lines) >= max_entries:
            lines.append("...")
            break
    return "\n".join(lines)


tree_text = summarize_tree(EXTRACTED_ROOT, max_depth=3)
print(tree_text[:6000])

folder_summary_df = (
    extracted_inventory_df
    .assign(top_folder=lambda df: df["relative_path"].apply(lambda p: Path(p).parts[0] if Path(p).parts else "root"))
    .groupby(["top_folder", "probable_source"], dropna=False)
    .agg(n_files=("file_path", "count"), size_bytes=("size_bytes", "sum"))
    .reset_index()
)
folder_summary_df["size_human"] = folder_summary_df["size_bytes"].apply(human_size)

extension_summary_df = (
    extracted_inventory_df
    .groupby(["extension", "probable_type"], dropna=False)
    .agg(n_files=("file_path", "count"), size_bytes=("size_bytes", "sum"))
    .reset_index()
    .sort_values("n_files", ascending=False)
)
extension_summary_df["size_human"] = extension_summary_df["size_bytes"].apply(human_size)

folder_summary_csv_path = RESULTS_ROOT / "E6_alkafri_folder_summary.csv"
extension_summary_csv_path = RESULTS_ROOT / "E6_alkafri_extension_summary.csv"
folder_summary_df.to_csv(folder_summary_csv_path, index=False)
extension_summary_df.to_csv(extension_summary_csv_path, index=False)

print("Folder summary:", folder_summary_csv_path)
print("Extension summary:", extension_summary_csv_path)
display(folder_summary_df)
display(extension_summary_df)

## Lectura preliminar DICOM / IMA

In [ ]:
print("ALKAFRI_ROOT:", ALKAFRI_ROOT)
print("RAW_ZIPS:", RAW_ZIPS)
print("EXTRACTED_ROOT:", EXTRACTED_ROOT)
print("EXTRACTED_ROOT existe:", EXTRACTED_ROOT.exists())

if EXTRACTED_ROOT.exists():
    all_files = [p for p in EXTRACTED_ROOT.rglob("*") if p.is_file()]
    print("Total archivos extraidos:", len(all_files))

    ext_counts = {}
    for p in all_files:
        ext = p.suffix.lower() if p.suffix else "<sin_extension>"
        ext_counts[ext] = ext_counts.get(ext, 0) + 1

    ext_df = pd.DataFrame(
        [{"extension": k, "n": v} for k, v in sorted(ext_counts.items(), key=lambda x: x[1], reverse=True)]
    )

    display(ext_df.head(30))

    print("\nEjemplos de archivos:")
    for p in all_files[:30]:
        print("-", p)
else:
    print("No existe EXTRACTED_ROOT")

In [ ]:
DICOM_TAGS = [
    "PatientID", "StudyInstanceUID", "SeriesInstanceUID", "SeriesDescription",
    "ProtocolName", "Modality", "ImageOrientationPatient", "ImagePositionPatient",
    "PixelSpacing", "SliceThickness", "Rows", "Columns", "InstanceNumber",
]


def safe_dcmread(path, stop_before_pixels=True):
    try:
        return pydicom.dcmread(str(path), stop_before_pixels=stop_before_pixels, force=True), None
    except Exception as exc:
        return None, repr(exc)


def get_dicom_tag(ds, tag_name):
    value = getattr(ds, tag_name, None)
    if value is None:
        return None
    try:
        if isinstance(value, pydicom.multival.MultiValue) or isinstance(value, (list, tuple)):
            return json.dumps([str(v) for v in value])
        return str(value)
    except Exception:
        return repr(value)


if "extracted_inventory_df" not in globals() or len(extracted_inventory_df) == 0:
    rows = []
    for p in EXTRACTED_ROOT.rglob("*"):
        if p.is_file():
            ext = p.suffix.lower() if p.suffix else "<sin_extension>"
            rel = p.relative_to(EXTRACTED_ROOT)
            rows.append({
                "file_path": str(p),
                "relative_path": str(rel),
                "file_name": p.name,
                "extension": ext,
                "parent_folder": str(p.parent.relative_to(EXTRACTED_ROOT)),
                "size_bytes": int(p.stat().st_size),
                "probable_source": probable_source_from_path(p) if "probable_source_from_path" in globals() else "unknown",
                "probable_type": probable_type_from_path(p) if "probable_type_from_path" in globals() else "unknown",
            })

    extracted_inventory_df = pd.DataFrame(rows)
    print("Inventario reconstruido:", len(extracted_inventory_df), "archivos")

if len(extracted_inventory_df) > 0:
    extracted_inventory_df["extension"] = extracted_inventory_df["extension"].astype(str).str.lower()
else:
    extracted_inventory_df = pd.DataFrame(columns=[
        "file_path", "relative_path", "file_name", "extension", "parent_folder", "size_bytes",
        "probable_source", "probable_type",
    ])

dicom_candidates_df = extracted_inventory_df[
    extracted_inventory_df["extension"].isin([".ima", ".dcm"])
].copy()

if len(dicom_candidates_df) == 0:
    possible_no_ext_df = extracted_inventory_df[
        extracted_inventory_df["extension"].isin(["<sin_extension>", "", "."])
    ].copy()

    print("No se detectaron .ima/.dcm por extension.")
    print("Archivos sin extension candidatos:", len(possible_no_ext_df))

    dicom_candidates_df = possible_no_ext_df.head(500).copy()

print("Candidatos DICOM/IMA:", len(dicom_candidates_df))

if len(dicom_candidates_df) > 0:
    display(dicom_candidates_df.head(20))
else:
    print("No hay candidatos DICOM/IMA. Revisar si los ZIPs fueron extraidos correctamente o si el dataset usa otro formato.")

dicom_sample_paths = dicom_candidates_df["file_path"].head(20).tolist() if len(dicom_candidates_df) else []

dicom_metadata_rows = []
for file_path in tqdm(dicom_sample_paths, desc="Leyendo muestra DICOM/IMA"):
    ds, error = safe_dcmread(file_path, stop_before_pixels=True)
    row = {
        "file_path": file_path,
        "relative_path": str(Path(file_path).relative_to(EXTRACTED_ROOT)),
        "read_ok": ds is not None,
        "read_error": error,
    }
    for tag in DICOM_TAGS:
        row[tag] = get_dicom_tag(ds, tag) if ds is not None else None
    dicom_metadata_rows.append(row)

dicom_metadata_sample_df = pd.DataFrame(dicom_metadata_rows)
expected_metadata_columns = ["file_path", "relative_path", "read_ok", "read_error", *DICOM_TAGS]
for col in expected_metadata_columns:
    if col not in dicom_metadata_sample_df.columns:
        dicom_metadata_sample_df[col] = None
dicom_metadata_sample_df = dicom_metadata_sample_df[expected_metadata_columns]

dicom_metadata_sample_csv_path = RESULTS_ROOT / "E6_alkafri_dicom_metadata_sample.csv"
dicom_metadata_sample_df.to_csv(dicom_metadata_sample_csv_path, index=False)

print("DICOM/IMA candidatos:", len(dicom_candidates_df))
print("Metadata sample:", dicom_metadata_sample_csv_path)
display(dicom_metadata_sample_df)

## Clasificacion preliminar axial/sagital

In [ ]:
def orientation_candidate_from_text_and_tags(row):
    path_text = str(row.get("relative_path", "")).lower()
    series_text = str(row.get("SeriesDescription", "") or "").lower()
    protocol_text = str(row.get("ProtocolName", "") or "").lower()
    orientation = row.get("ImageOrientationPatient")

    for label, text, reason in [
        ("axial_candidate", path_text, "path_keyword"),
        ("axial_candidate", series_text, "series_description_keyword"),
        ("axial_candidate", protocol_text, "protocol_keyword"),
    ]:
        if "ax" in text or "axial" in text or "tra" in text:
            return label, reason

    for label, text, reason in [
        ("sagittal_candidate", path_text, "path_keyword"),
        ("sagittal_candidate", series_text, "series_description_keyword"),
        ("sagittal_candidate", protocol_text, "protocol_keyword"),
    ]:
        if "sag" in text or "sagittal" in text:
            return label, reason

    if orientation not in [None, "", "None"]:
        try:
            if isinstance(orientation, str):
                values = np.asarray(json.loads(orientation), dtype=float)
            else:
                values = np.asarray(orientation, dtype=float)

            row_cosine = values[:3]
            col_cosine = values[3:]
            normal = np.abs(np.cross(row_cosine, col_cosine))
            dominant = int(np.argmax(normal))

            if dominant == 2:
                return "axial_candidate", "orientation_inference"
            if dominant == 0:
                return "sagittal_candidate", "orientation_inference"
        except Exception:
            pass

    return "unknown", "unknown"


orientation_rows = []

expected_columns = [
    "file_path",
    "relative_path",
    "extension",
    "read_ok",
    "read_error",
    *DICOM_TAGS,
    "orientation_candidate",
    "classification_reason",
]

if len(dicom_candidates_df) == 0:
    print("No hay candidatos DICOM/IMA para clasificar. Se exporta CSV vacio con columnas esperadas.")
    series_orientation_df = pd.DataFrame(columns=expected_columns)
else:
    for _, inv_row in tqdm(
        dicom_candidates_df.iterrows(),
        total=len(dicom_candidates_df),
        desc="Clasificando DICOM/IMA",
    ):
        file_path = inv_row["file_path"]
        ds, error = safe_dcmread(file_path, stop_before_pixels=True)

        base = {
            "file_path": file_path,
            "relative_path": inv_row.get("relative_path", ""),
            "extension": inv_row.get("extension", ""),
            "read_ok": ds is not None,
            "read_error": error,
        }

        for tag in DICOM_TAGS:
            base[tag] = get_dicom_tag(ds, tag) if ds is not None else None

        candidate, reason = orientation_candidate_from_text_and_tags(base)
        base["orientation_candidate"] = candidate
        base["classification_reason"] = reason
        orientation_rows.append(base)

    series_orientation_df = pd.DataFrame(orientation_rows)

    for col in expected_columns:
        if col not in series_orientation_df.columns:
            series_orientation_df[col] = None
    series_orientation_df = series_orientation_df[expected_columns]

series_orientation_csv_path = RESULTS_ROOT / "E6_alkafri_series_orientation_candidates.csv"
series_orientation_df.to_csv(series_orientation_csv_path, index=False)

print("Orientaciones:", series_orientation_csv_path)

if len(series_orientation_df) > 0:
    display(series_orientation_df["orientation_candidate"].value_counts(dropna=False).rename_axis("candidate").reset_index(name="n"))
    display(series_orientation_df.head())
else:
    print("series_orientation_df vacio.")

## Inventario de ground truth

In [ ]:
GT_KEYWORDS = [
    "disc", "posterior", "thecal", "canal", "intervertebral",
    "l3", "l4", "l5", "s1", "mask", "label", "ground", "gt", "segmentation",
]


def gt_keyword_flags(text):
    text_lower = str(text).lower()
    return {f"has_{kw}": bool(kw in text_lower) for kw in GT_KEYWORDS}


gt_df = extracted_inventory_df[
    (extracted_inventory_df["probable_source"].eq("ground_truth"))
    | extracted_inventory_df["has_ground"]
    | extracted_inventory_df["has_label"]
    | extracted_inventory_df["has_mask"]
    | extracted_inventory_df["has_gt"]
].copy()

gt_rows = []
for _, row in gt_df.iterrows():
    text = f"{row['relative_path']} {row['file_name']}"
    item = row.to_dict()
    item.update(gt_keyword_flags(text))
    if row["extension"] == ".mat":
        try:
            item["mat_variables"] = json.dumps([var[0] for var in whosmat(row["file_path"])], ensure_ascii=False)
        except Exception as exc:
            item["mat_variables"] = None
            item["mat_read_error"] = repr(exc)
    gt_rows.append(item)

ground_truth_inventory_df = pd.DataFrame(gt_rows)
ground_truth_inventory_csv_path = RESULTS_ROOT / "E6_alkafri_ground_truth_inventory.csv"
ground_truth_inventory_df.to_csv(ground_truth_inventory_csv_path, index=False)

print("Ground truth files:", len(ground_truth_inventory_df))
print("Ground truth inventory:", ground_truth_inventory_csv_path)
if len(ground_truth_inventory_df) > 0:
    display(ground_truth_inventory_df.head(30))
else:
    print("No se detectaron archivos de ground truth por fuente o keywords.")

## Visualizacion preliminar de imagenes axiales nativas

In [ ]:
def normalize_percentile(image, p_low=1, p_high=99):
    arr = image.astype(np.float32)
    low, high = np.percentile(arr, [p_low, p_high])
    if np.isclose(low, high):
        return np.zeros_like(arr, dtype=np.float32)
    return np.clip((arr - low) / (high - low), 0, 1).astype(np.float32)


def read_dicom_pixels(path):
    ds = pydicom.dcmread(str(path), force=True)
    arr = ds.pixel_array.astype(np.float32)
    return ds, arr


axial_image_paths = []
visualization_errors = []
overlay_preliminary_path = None
overlay_pairing_status = "No se pudo emparejar imagen/mascara sin ambiguedad en este inventario; queda para notebook 13."

if "orientation_candidate" in series_orientation_df.columns and len(series_orientation_df) > 0:
    axial_candidates_df = series_orientation_df[series_orientation_df["orientation_candidate"].eq("axial_candidate")].copy()
    if len(axial_candidates_df) == 0 and "read_ok" in series_orientation_df.columns:
        axial_candidates_df = series_orientation_df[series_orientation_df["read_ok"].fillna(False)].copy()
        print("No se detectaron candidatos axiales explicitos; se usara muestra DICOM legible como fallback visual.")
else:
    axial_candidates_df = pd.DataFrame(columns=series_orientation_df.columns if "series_orientation_df" in globals() else [])
    print("No hay series DICOM/IMA clasificadas para visualizacion axial.")

for idx, row in enumerate(axial_candidates_df.head(5).itertuples(index=False), start=1):
    output_path = FIGURES_ROOT / f"E6_alkafri_axial_image_example_{idx:02d}.png"
    try:
        ds, pixels = read_dicom_pixels(row.file_path)
        fig, ax = plt.subplots(figsize=(5, 5))
        ax.imshow(normalize_percentile(pixels), cmap="gray")
        ax.set_title(f"Al-Kafri axial candidate {idx}\n{Path(row.relative_path).name}")
        ax.axis("off")
        fig.tight_layout()
        fig.savefig(output_path, dpi=160, bbox_inches="tight")
        plt.close(fig)
        axial_image_paths.append(str(output_path))
    except Exception as exc:
        visualization_errors.append({"file_path": getattr(row, "file_path", None), "error": repr(exc)})

if len(axial_image_paths) == 0:
    image_format_df = extracted_inventory_df[
        extracted_inventory_df["extension"].isin([".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"])
    ].copy()
    print("No se pudo visualizar DICOM axial. Imagenes raster encontradas como alternativa:", len(image_format_df))
    for idx, row in enumerate(image_format_df.head(5).itertuples(index=False), start=1):
        output_path = FIGURES_ROOT / f"E6_alkafri_axial_image_example_{idx:02d}.png"
        try:
            image = np.asarray(Image.open(row.file_path))
            fig, ax = plt.subplots(figsize=(5, 5))
            ax.imshow(normalize_percentile(image), cmap="gray")
            ax.set_title(f"Al-Kafri image fallback {idx}\n{Path(row.relative_path).name}")
            ax.axis("off")
            fig.tight_layout()
            fig.savefig(output_path, dpi=160, bbox_inches="tight")
            plt.close(fig)
            axial_image_paths.append(str(output_path))
        except Exception as exc:
            visualization_errors.append({"file_path": getattr(row, "file_path", None), "error": repr(exc)})

mat_files_df = extracted_inventory_df[extracted_inventory_df["extension"].eq(".mat")].copy()
print("Archivos .mat detectados para analisis posterior:", len(mat_files_df))

print("Figuras axiales exportadas:")
for path in axial_image_paths:
    print(path)
if visualization_errors:
    print("Errores visualizacion:", visualization_errors[:5])
print("Overlay preliminar:", overlay_pairing_status)

## Revision de Source_Code

In [ ]:
SOURCE_KEYWORDS = ["dicom", "mask", "label", "ground", "disc", "thecal", "posterior", "l3", "l4", "l5"]
source_code_df = extracted_inventory_df[
    extracted_inventory_df["probable_source"].eq("source_code")
    & extracted_inventory_df["extension"].eq(".m")
].copy()

source_hits = []
for _, row in source_code_df.iterrows():
    path = Path(row["file_path"])
    try:
        text = path.read_text(encoding="utf-8", errors="ignore").lower()
    except Exception as exc:
        source_hits.append({
            "file_path": str(path),
            "relative_path": row["relative_path"],
            "read_ok": False,
            "read_error": repr(exc),
        })
        continue
    hit = {
        "file_path": str(path),
        "relative_path": row["relative_path"],
        "read_ok": True,
        "read_error": None,
    }
    for keyword in SOURCE_KEYWORDS:
        hit[f"count_{keyword}"] = int(text.count(keyword))
    source_hits.append(hit)

source_code_keyword_hits_df = pd.DataFrame(source_hits)
source_code_hits_csv_path = RESULTS_ROOT / "E6_alkafri_source_code_keyword_hits.csv"
source_code_keyword_hits_df.to_csv(source_code_hits_csv_path, index=False)

print("Archivos .m:", len(source_code_df))
print("Source code keyword hits:", source_code_hits_csv_path)
display(source_code_keyword_hits_df.head())

## Revision de Radiologists Notes

In [ ]:
notes_df = extracted_inventory_df[extracted_inventory_df["probable_source"].eq("radiologist_notes")].copy()
notes_rows = []
for _, row in notes_df.iterrows():
    path = Path(row["file_path"])
    item = row.to_dict()
    item["preview"] = None
    item["preview_error"] = None
    if row["extension"] in {".txt", ".csv"}:
        try:
            item["preview"] = path.read_text(encoding="utf-8", errors="ignore")[:1000]
        except Exception as exc:
            item["preview_error"] = repr(exc)
    elif row["extension"] in {".xlsx", ".xls"}:
        try:
            preview_df = pd.read_excel(path, nrows=5)
            item["preview"] = preview_df.to_json(orient="records", force_ascii=False)
        except Exception as exc:
            item["preview_error"] = repr(exc)
    notes_rows.append(item)

radiologist_notes_inventory_df = pd.DataFrame(notes_rows)
radiologist_notes_inventory_csv_path = RESULTS_ROOT / "E6_alkafri_radiologist_notes_inventory.csv"
radiologist_notes_inventory_df.to_csv(radiologist_notes_inventory_csv_path, index=False)

print("Radiologist notes files:", len(radiologist_notes_inventory_df))
print("Radiologist notes inventory:", radiologist_notes_inventory_csv_path)
display(radiologist_notes_inventory_df.head())

## Reporte JSON

In [ ]:
extension_counts = extracted_inventory_df["extension"].value_counts().to_dict() if "extension" in extracted_inventory_df.columns and len(extracted_inventory_df) else {}
probable_type_counts = extracted_inventory_df["probable_type"].value_counts().to_dict() if "probable_type" in extracted_inventory_df.columns and len(extracted_inventory_df) else {}
orientation_counts = (
    series_orientation_df["orientation_candidate"].value_counts().to_dict()
    if "orientation_candidate" in series_orientation_df.columns and len(series_orientation_df)
    else {}
)

dicom_candidates_detected = int(len(dicom_candidates_df)) if "dicom_candidates_df" in globals() else 0
dicom_orientation_classification_available = bool(
    dicom_candidates_detected > 0
    and "orientation_candidate" in series_orientation_df.columns
    and len(series_orientation_df) > 0
)

exports = {
    "zip_files": str(zip_files_csv_path),
    "zip_inventory": str(zip_inventory_csv_path),
    "extraction_report": str(extraction_report_csv_path),
    "extracted_file_inventory": str(extracted_inventory_csv_path),
    "folder_summary": str(folder_summary_csv_path),
    "extension_summary": str(extension_summary_csv_path),
    "dicom_metadata_sample": str(dicom_metadata_sample_csv_path),
    "series_orientation_candidates": str(series_orientation_csv_path),
    "ground_truth_inventory": str(ground_truth_inventory_csv_path),
    "source_code_keyword_hits": str(source_code_hits_csv_path),
    "radiologist_notes_inventory": str(radiologist_notes_inventory_csv_path),
    "axial_figures": axial_image_paths,
    "overlay_preliminary": overlay_preliminary_path,
}

if dicom_candidates_detected == 0:
    recommendation_notebook_13 = "Revisar estructura extraida y adaptar notebook 13 al formato real del dataset"
else:
    recommendation_notebook_13 = (
        "Construir un notebook de preprocesamiento axial que agrupe series por PatientID/StudyInstanceUID/SeriesInstanceUID, "
        "confirme orientaciones axiales nativas, defina reglas de emparejamiento imagen-ground truth y exporte un dataset axial normalizado."
    )

report = {
    "detected_zips": zip_files_df[["zip_name", "zip_path", "size_bytes", "size_human"]].to_dict(orient="records"),
    "expected_zips_all_detected": len(missing_expected) == 0,
    "total_extracted_files": int(len(extracted_inventory_df)),
    "count_ima": int(extension_counts.get(".ima", 0)),
    "count_dcm": int(extension_counts.get(".dcm", 0)),
    "dicom_candidates_detected": dicom_candidates_detected,
    "dicom_orientation_classification_available": dicom_orientation_classification_available,
    "count_possible_masks_or_labels": int(probable_type_counts.get("mask_or_label_image", 0) + len(ground_truth_inventory_df)),
    "count_mat": int(extension_counts.get(".mat", 0)),
    "count_axial_candidate_files": int(orientation_counts.get("axial_candidate", 0)),
    "count_sagittal_candidate_files": int(orientation_counts.get("sagittal_candidate", 0)),
    "count_unknown_orientation_files": int(orientation_counts.get("unknown", 0)),
    "dicom_read_success": bool(series_orientation_df["read_ok"].fillna(False).any()) if "read_ok" in series_orientation_df.columns and len(series_orientation_df) else False,
    "visualized_at_least_one_axial_image": bool(len(axial_image_paths) > 0),
    "detected_ground_truth": bool(len(ground_truth_inventory_df) > 0),
    "paired_image_mask": bool(overlay_preliminary_path is not None),
    "pairing_status": overlay_pairing_status,
    "exports": exports,
    "recommendation": recommendation_notebook_13,
    "recommendation_for_notebook_13": recommendation_notebook_13,
}

report_json_path = RESULTS_ROOT / "E6_alkafri_inventory_report.json"
with open(report_json_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

print("Reporte JSON:", report_json_path)
print(json.dumps(report, indent=2, ensure_ascii=False)[:5000])

## Conclusion Markdown

In [ ]:
zip_table_md = zip_files_df.to_markdown(index=False)
inventory_table_md = zip_inventory_df.to_markdown(index=False)
extension_table_md = extension_summary_df.head(30).to_markdown(index=False)
orientation_table_md = (
    series_orientation_df["orientation_candidate"].value_counts(dropna=False)
    .rename_axis("orientation_candidate")
    .reset_index(name="n")
    .to_markdown(index=False)
    if "orientation_candidate" in series_orientation_df.columns and len(series_orientation_df) else "Sin DICOM/IMA detectados o clasificables."
)
gt_table_md = (
    ground_truth_inventory_df[["relative_path", "extension", "probable_type"]].head(30).to_markdown(index=False)
    if len(ground_truth_inventory_df) and all(col in ground_truth_inventory_df.columns for col in ["relative_path", "extension", "probable_type"])
    else "No se detecto ground truth por inventario o keywords."
)

conclusion_text = f'''# Conclusion tecnica - E6 Inventario Al-Kafri/Sudirman

## Objetivo

Realizar inventario, descompresion controlada, analisis de estructura y lectura preliminar del dataset axial Al-Kafri/Sudirman, sin entrenar modelos.

## Por que se incorpora Al-Kafri/Sudirman

El spike axial sobre SPIDER mostro que las vistas axiales reconstruidas desde ese dataset pueden no ser visualmente adecuadas en varios casos por anisotropia, baja resolucion efectiva o geometria no nativa axial. Al-Kafri/Sudirman se incorpora como dataset complementario porque contiene cortes axiales nativos de RM lumbar y puede servir para el modulo axial del MVP.

## Archivos cargados

{zip_table_md}

## Inventario interno de ZIPs

{inventory_table_md}

## Estructura y formatos encontrados

{extension_table_md}

## Lectura DICOM preliminar

- Candidatos DICOM/IMA detectados: {report["dicom_candidates_detected"]}.
- Archivos `.ima`: {report["count_ima"]}.
- Archivos `.dcm`: {report["count_dcm"]}.
- Clasificacion de orientacion disponible: {report["dicom_orientation_classification_available"]}.
- Lectura DICOM exitosa en al menos un archivo: {report["dicom_read_success"]}.

## Hallazgos sobre cortes axiales

{orientation_table_md}

Se exportaron figuras axiales preliminares cuando fue posible: {axial_image_paths}.

## Hallazgos sobre ground truth

{gt_table_md}

Ground truth detectado: {report["detected_ground_truth"]}.

Emparejamiento imagen/mascara: {overlay_pairing_status}

## Viabilidad preliminar

El dataset es viable para continuar si se confirma el formato real de imagenes, existen candidatos axiales nativos y el ground truth puede organizarse por paciente/serie/nivel. Si no se detectan DICOM/IMA por extension, el siguiente paso debe adaptar el preprocesamiento al formato efectivamente encontrado en el inventario extraido.

## Limitaciones

- No se entrenan modelos.
- No se asume estructura sin verificar.
- No se ejecuta codigo MATLAB.
- Las notas de radiologos se inventarian, pero no se usan para entrenamiento.
- El emparejamiento imagen/mascara queda pendiente si no hay una regla inequívoca.
- No se realizan diagnosticos ni mediciones clinicas.

## Siguiente paso recomendado

{recommendation_notebook_13}
'''

conclusion_path = DOCS_ROOT / "E6_alkafri_inventory_conclusion.md"
with open(conclusion_path, "w", encoding="utf-8") as f:
    f.write(conclusion_text)

print("Conclusion Markdown:", conclusion_path)

## Criterio de aceptacion

Este notebook cumple el objetivo si:

- Detecta los 4 ZIPs esperados.
- Inventaria el contenido de cada ZIP antes de extraer.
- Descomprime controladamente en `extracted`.
- Identifica imagenes DICOM/IMA.
- Intenta leer metadatos DICOM.
- Identifica candidatos axiales.
- Identifica ground truth.
- Genera visualizaciones axiales nativas cuando es posible.
- Exporta CSVs, JSON, Markdown y figuras.
- No entrena ningun modelo.